In [1]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [2]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

list_of_files = glob.glob('data/csv/*.csv')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = pd.read_csv(f, sep=',', header=0, index_col=0)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Convert the 'date_of_prediction' column to datetime format
final_df['date_of_prediction'] = pd.to_datetime(final_df['date_of_prediction'])
final_df['month_of_prediction'] = final_df['date_of_prediction'].dt.month
final_df = final_df.drop(columns=['date_of_prediction', 'realization_year'])


In [3]:
# Calculating all the statistics for each region and model

# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_4'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction']).agg(['mean', 'std']).reset_index()
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction'], right_on=['region', 'model', 'season', 'month_of_prediction'], how='left').dropna()

# Calculating metrics
stat_clean['potential_skill'] = np.square(stat_clean['corr'])
stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

# dropping unnecessary columns
stat_clean = stat_clean.drop(['pred_mean','pred_std','actual_mean','actual_std'], axis = 1)


In [7]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='potential_skill')
    
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Reds', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=True, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/potential_skill.png')
plt.close()

In [8]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='conditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=True, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Conditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/conditional_bias.png')
plt.close()